In [2]:
import torch
from torch import nn
from transformers import BertModel,BertTokenizer

In [3]:
model_name = 'bert-base-uncased'

tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name,output_hidden_states=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
text = "After stealing money from the bank vault, the bank robber was seen " \
       "fishing on the Mississippi river bank."

In [12]:
token_input = tokenizer(text,return_tensors='pt')

In [13]:
token_input

{'input_ids': tensor([[  101,  2044, 11065,  2769,  2013,  1996,  2924, 11632,  1010,  1996,
          2924, 27307,  2001,  2464,  5645,  2006,  1996,  5900,  2314,  2924,
          1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [14]:
token_input['input_ids'],token_input['input_ids'].shape

(tensor([[  101,  2044, 11065,  2769,  2013,  1996,  2924, 11632,  1010,  1996,
           2924, 27307,  2001,  2464,  5645,  2006,  1996,  5900,  2314,  2924,
           1012,   102]]),
 torch.Size([1, 22]))

# 其中由于输入文本仅一句,因此batch为1

# 前向过程
- forward
- - embedding => encoder => pooler  

In [15]:
model.eval()
with torch.no_grad():
  outputs = model(**token_input)

### 3. output / 输出

* len(outputs) == 3
  *(outputs 列表的长度为 3)*

* outputs[0]
    * last_hidden_state, shape: batch_size * seq_len * hidden_size(1 * 22 * 768)
      *(`last_hidden_state` 最后一层的隐藏状态，形状：`batch_size * seq_len * hidden_size` 即 1 * 22 * 768)*

* outputs[1]
    * pooler_output, shape: batch_size * hidden_size(1 * 768)
      *(`pooler_output` 池化层输出，形状：`batch_size * hidden_size` 即 1 * 768)*
    * Last layer hidden-state of the first token of the sequence (classification token, [CLS])
  *(序列第一个词元（分类词元，即 [CLS]）在最后一层的隐藏状态)*

* outputs[2] (model.config.output_hidden_states = True)
  *(当设置 `model.config.output_hidden_states = True` 时才会输出)*
    * type: tuple
      *(类型：元组)*
    * one for the output of the embeddings(1), if the model has an embedding layer, + one for the output of each layer
      *(1 个用于嵌入层的输出——如果模型包含嵌入层的话，加上每一层的输出各 1 个)*
        * (1+12) * (batch_size * seq_len * hidden_size) = 1 * 22 * 768
          *(即 1 层嵌入层输出 + 12 层隐藏层输出，总共 13 个张量，每个张量的形状为 13 * 22 * 768)*

outputs[0] == outputs[2][-1]

In [16]:
len(outputs)

3

In [17]:
type(outputs[2]),len(outputs[2])

(tuple, 13)

In [18]:
outputs[0]

tensor([[[-0.4964, -0.1831, -0.5231,  ..., -0.1902,  0.3738,  0.3964],
         [-0.1323, -0.2762, -0.3495,  ..., -0.4567,  0.3786, -0.1096],
         [-0.3626, -0.4002,  0.0676,  ..., -0.3207, -0.2709, -0.3004],
         ...,
         [ 0.2961, -0.2856, -0.0382,  ..., -0.6056, -0.5163,  0.2005],
         [ 0.4878, -0.0909, -0.2358,  ..., -0.0017, -0.5945, -0.2431],
         [-0.2517, -0.3519, -0.4688,  ...,  0.2500,  0.0336, -0.2627]]])

In [19]:
outputs[2][-1]

tensor([[[-0.4964, -0.1831, -0.5231,  ..., -0.1902,  0.3738,  0.3964],
         [-0.1323, -0.2762, -0.3495,  ..., -0.4567,  0.3786, -0.1096],
         [-0.3626, -0.4002,  0.0676,  ..., -0.3207, -0.2709, -0.3004],
         ...,
         [ 0.2961, -0.2856, -0.0382,  ..., -0.6056, -0.5163,  0.2005],
         [ 0.4878, -0.0909, -0.2358,  ..., -0.0017, -0.5945, -0.2431],
         [-0.2517, -0.3519, -0.4688,  ...,  0.2500,  0.0336, -0.2627]]])

In [20]:
outputs[0] == outputs[2][-1]

tensor([[[True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         ...,
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True]]])

In [21]:
outputs[1]

tensor([[-0.6031, -0.3342, -0.7174,  0.3347,  0.5145, -0.1722,  0.4502,  0.2768,
         -0.3769, -0.9998, -0.3657,  0.7535,  0.9817, -0.0192,  0.7959, -0.3459,
         -0.1338, -0.3026,  0.1097,  0.5836,  0.5736,  0.9999,  0.1798,  0.1845,
          0.2250,  0.9109, -0.5653,  0.8616,  0.8994,  0.7423, -0.2525,  0.0394,
         -0.9894, -0.1331, -0.7763, -0.9826,  0.2223, -0.6115,  0.1941,  0.0177,
         -0.7634,  0.2312,  0.9999, -0.7000,  0.4623, -0.2202, -1.0000,  0.1908,
         -0.8150,  0.6483,  0.5878,  0.8198,  0.1014,  0.3185,  0.3963, -0.3216,
         -0.1701,  0.0588, -0.1544, -0.4987, -0.5284,  0.1228, -0.4823, -0.7788,
          0.6954,  0.0891, -0.0855, -0.1500,  0.0390, -0.0760,  0.6154,  0.2662,
         -0.0129, -0.7253,  0.1352,  0.2921, -0.5613,  1.0000,  0.1536, -0.9681,
          0.7166,  0.2600,  0.4519,  0.5470, -0.2798, -1.0000,  0.3419, -0.2645,
         -0.9863,  0.1263,  0.5249, -0.2000,  0.5980,  0.4752, -0.2355, -0.4808,
         -0.3786, -0.7284, -

In [22]:
outputs[2][-1]

tensor([[[-0.4964, -0.1831, -0.5231,  ..., -0.1902,  0.3738,  0.3964],
         [-0.1323, -0.2762, -0.3495,  ..., -0.4567,  0.3786, -0.1096],
         [-0.3626, -0.4002,  0.0676,  ..., -0.3207, -0.2709, -0.3004],
         ...,
         [ 0.2961, -0.2856, -0.0382,  ..., -0.6056, -0.5163,  0.2005],
         [ 0.4878, -0.0909, -0.2358,  ..., -0.0017, -0.5945, -0.2431],
         [-0.2517, -0.3519, -0.4688,  ...,  0.2500,  0.0336, -0.2627]]])

In [23]:
model.pooler(outputs[2][-1])

tensor([[-0.6031, -0.3342, -0.7174,  0.3347,  0.5145, -0.1722,  0.4502,  0.2768,
         -0.3769, -0.9998, -0.3657,  0.7535,  0.9817, -0.0192,  0.7959, -0.3459,
         -0.1338, -0.3026,  0.1097,  0.5836,  0.5736,  0.9999,  0.1798,  0.1845,
          0.2250,  0.9109, -0.5653,  0.8616,  0.8994,  0.7423, -0.2525,  0.0394,
         -0.9894, -0.1331, -0.7763, -0.9826,  0.2223, -0.6115,  0.1941,  0.0177,
         -0.7634,  0.2312,  0.9999, -0.7000,  0.4623, -0.2202, -1.0000,  0.1908,
         -0.8150,  0.6483,  0.5878,  0.8198,  0.1014,  0.3185,  0.3963, -0.3216,
         -0.1701,  0.0588, -0.1544, -0.4987, -0.5284,  0.1228, -0.4823, -0.7788,
          0.6954,  0.0891, -0.0855, -0.1500,  0.0390, -0.0760,  0.6154,  0.2662,
         -0.0129, -0.7253,  0.1352,  0.2921, -0.5613,  1.0000,  0.1536, -0.9681,
          0.7166,  0.2600,  0.4519,  0.5470, -0.2798, -1.0000,  0.3419, -0.2645,
         -0.9863,  0.1263,  0.5249, -0.2000,  0.5980,  0.4752, -0.2355, -0.4808,
         -0.3786, -0.7284, -

In [24]:
model.pooler(outputs[2][-1]) == outputs[1]

tensor([[True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, T

In [25]:
token_input

{'input_ids': tensor([[  101,  2044, 11065,  2769,  2013,  1996,  2924, 11632,  1010,  1996,
          2924, 27307,  2001,  2464,  5645,  2006,  1996,  5900,  2314,  2924,
          1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [26]:
model.embeddings(token_input['input_ids'],token_input['token_type_ids'])

tensor([[[ 0.1686, -0.2858, -0.3261,  ..., -0.0276,  0.0383,  0.1640],
         [ 0.2329,  0.1390,  0.2979,  ..., -0.0655,  0.8885,  0.5109],
         [ 0.2257, -0.7165, -0.7255,  ...,  0.4844,  0.6030, -0.0957],
         ...,
         [-0.0374, -0.6155, -1.4419,  ...,  0.0793, -0.0811, -0.3802],
         [-0.0228,  0.4207, -0.3288,  ...,  0.4464,  0.5178,  0.5501],
         [-0.2350,  0.1566, -0.0462,  ..., -0.4206,  0.3074, -0.2288]]],
       grad_fn=<NativeLayerNormBackward0>)

In [27]:
outputs[2][0]

tensor([[[ 0.1686, -0.2858, -0.3261,  ..., -0.0276,  0.0383,  0.1640],
         [ 0.2329,  0.1390,  0.2979,  ..., -0.0655,  0.8885,  0.5109],
         [ 0.2257, -0.7165, -0.7255,  ...,  0.4844,  0.6030, -0.0957],
         ...,
         [-0.0374, -0.6155, -1.4419,  ...,  0.0793, -0.0811, -0.3802],
         [-0.0228,  0.4207, -0.3288,  ...,  0.4464,  0.5178,  0.5501],
         [-0.2350,  0.1566, -0.0462,  ..., -0.4206,  0.3074, -0.2288]]])

In [28]:
model.embeddings(token_input['input_ids'],token_input['token_type_ids']) ==  outputs[2][0]

tensor([[[True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         ...,
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True]]])

### 3. output

* len(outputs) == 3
* outputs[0]
    * last_hidden_state, shape: batch_size\*seq_len\*hidden_size(1\*22\*768)
* outputs[1]
    * pooler_output, shape: batch_size\*hidden_size(1\*768)
    * Last layer hidden-state of the first token of the sequence (classification token, [CLS])
* outputs[2] (model.config.output_hidden_states = True)
    * type: tuple
    * one for the output of the embeddings(1), if the model has an embedding layer(12), + one for the output of each layer)
        * (1+12)\*(batch_size\*seq_len\*hidden_size) = 13\*1\*22\*768

* outputs[0] == outputs[2][-1]
* outputs[1] == model.pooler(outputs[2][-1])
* outputs[2][0] == model.embeddings(token_input['input_ids'], token_input['token_type_ids'])

In [29]:
for i in range(len(outputs[2])):
  print(i,outputs[2][i].shape)

0 torch.Size([1, 22, 768])
1 torch.Size([1, 22, 768])
2 torch.Size([1, 22, 768])
3 torch.Size([1, 22, 768])
4 torch.Size([1, 22, 768])
5 torch.Size([1, 22, 768])
6 torch.Size([1, 22, 768])
7 torch.Size([1, 22, 768])
8 torch.Size([1, 22, 768])
9 torch.Size([1, 22, 768])
10 torch.Size([1, 22, 768])
11 torch.Size([1, 22, 768])
12 torch.Size([1, 22, 768])
